In [4]:
from datetime import datetime, date
from typing import List, Dict, Optional
from enum import Enum

class EstadoPedido(Enum):
    PENDIENTE = "pendiente"
    EN_PREPARACION = "en_preparacion"
    ENTREGADO = "entregado"
    CANCELADO = "cancelado"

class Persona:
    def __init__(self, nombre: str, identificacion: str, telefono: str):
        self.nombre = nombre
        self.identificacion = identificacion
        self.telefono = telefono

    def actualizar_datos(self, nombre: str = None, telefono: str = None):
        if nombre:
            self.nombre = nombre
        if telefono:
            self.telefono = telefono

class Cliente(Persona):
    def __init__(self, nombre: str, identificacion: str, telefono: str):
        super().__init__(nombre, identificacion, telefono)
        self.historial_pedidos: List[Pedido] = []
        self.puntos_fidelidad: int = 0

    def realizar_pedido(self, pedido: 'Pedido') -> bool:
        self.historial_pedidos.append(pedido)
        return True

    def consultar_historial(self) -> List['Pedido']:
        return self.historial_pedidos

    def obtener_puntos(self) -> int:
        return self.puntos_fidelidad

class Empleado(Persona):
    def __init__(self, nombre: str, identificacion: str, telefono: str, rol: str, horario: str, salario: float):
        super().__init__(nombre, identificacion, telefono)
        self.rol = rol
        self.horario = horario
        self.salario = salario

    def actualizar_inventario(self, inventario: 'Inventario', producto: str, cantidad: int) -> bool:
        return inventario.actualizar_cantidad(producto, cantidad)

    def gestionar_pedidos(self, pedido: 'Pedido', nuevo_estado: EstadoPedido) -> bool:
        return pedido.actualizar_estado(nuevo_estado)

    def aplicar_promociones(self, pedido: 'Pedido', promocion: 'Promocion') -> bool:
        return promocion.aplicar_descuento(pedido)

class ProductoBase:
    def __init__(self, nombre: str, precio: float, descripcion: str, categoria: str):
        self.nombre = nombre
        self.precio = precio
        self.descripcion = descripcion
        self.categoria = categoria
        self.disponible = True

    def obtener_precio(self) -> float:
        return self.precio

    def actualizar_disponibilidad(self, disponible: bool):
        self.disponible = disponible

class Bebida(ProductoBase):
    def __init__(self, nombre: str, precio: float, descripcion: str, 
                 tamaño: str, tipo: str, cafeina: bool):
        super().__init__(nombre, precio, descripcion, "bebida")
        self.tamaño = tamaño
        self.tipo = tipo
        self.cafeina = cafeina
        self.opciones_personalizacion: List[str] = []

    def personalizar_bebida(self, opciones: List[str]) -> bool:
        self.opciones_personalizacion = opciones
        return True

    def calcular_precio_final(self) -> float:
        precio_final = self.precio
        for opcion in self.opciones_personalizacion:
            if opcion == "leche_almendra":
                precio_final += 1.0
            elif opcion == "extra_shot":
                precio_final += 0.5
        return precio_final

class Postre(ProductoBase):
    def __init__(self, nombre: str, precio: float, descripcion: str, 
                 vegano: bool, sin_gluten: bool):
        super().__init__(nombre, precio, descripcion, "postre")
        self.vegano = vegano
        self.sin_gluten = sin_gluten
        self.ingredientes: List[str] = []

    def verificar_alergenos(self) -> List[str]:
        alergenos = []
        for ingrediente in self.ingredientes:
            if ingrediente in ["gluten", "leche", "huevos", "nueces"]:
                alergenos.append(ingrediente)
        return alergenos

    def obtener_informacion_nutricional(self) -> Dict[str, float]:
        return {
            "calorias": 0.0,  # Estos valores deberían ser reales
            "proteinas": 0.0,
            "carbohidratos": 0.0,
            "grasas": 0.0
        }

class Inventario:
    def __init__(self):
        self.ingredientes: Dict[str, int] = {}
        self.stock_minimo: int = 10

    def verificar_stock(self, ingrediente: str) -> bool:
        return self.ingredientes.get(ingrediente, 0) > self.stock_minimo

    def actualizar_cantidad(self, ingrediente: str, cantidad: int) -> bool:
        if ingrediente in self.ingredientes:
            nueva_cantidad = self.ingredientes[ingrediente] + cantidad
            if nueva_cantidad >= 0:
                self.ingredientes[ingrediente] = nueva_cantidad
                return True
        return False

    def alerta_stock_bajo(self) -> List[str]:
        return [ingrediente for ingrediente, cantidad in self.ingredientes.items() 
                if cantidad <= self.stock_minimo]

class Pedido:
    def __init__(self, cliente: Cliente):
        self.numero_pedido = datetime.now().strftime("%Y%m%d%H%M%S")
        self.fecha = datetime.now()
        self.estado = EstadoPedido.PENDIENTE
        self.items: List[ProductoBase] = []
        self.total: float = 0.0
        self.cliente = cliente

    def calcular_total(self) -> float:
        self.total = sum(item.obtener_precio() for item in self.items)
        return self.total

    def actualizar_estado(self, nuevo_estado: EstadoPedido) -> bool:
        self.estado = nuevo_estado
        return True

    def agregar_producto(self, producto: ProductoBase) -> bool:
        self.items.append(producto)
        self.calcular_total()
        return True

    def personalizar_producto(self, producto: Bebida, opciones: List[str]) -> bool:
        if producto in self.items and isinstance(producto, Bebida):
            return producto.personalizar_bebida(opciones)
        return False

class Promocion:
    def __init__(self, codigo: str, descuento: float, validez_inicio: date, 
                 validez_fin: date, puntos_requeridos: int):
        self.codigo = codigo
        self.descuento = descuento
        self.validez_inicio = validez_inicio
        self.validez_fin = validez_fin
        self.puntos_requeridos = puntos_requeridos

    def aplicar_descuento(self, pedido: Pedido) -> bool:
        if self.verificar_validez() and pedido.cliente.puntos_fidelidad >= self.puntos_requeridos:
            pedido.total *= (1 - self.descuento)
            return True
        return False

    def verificar_validez(self) -> bool:
        hoy = date.today()
        return self.validez_inicio <= hoy <= self.validez_fin

    def calcular_puntos(self, total_pedido: float) -> int:
        return int(total_pedido // 10)  # 1 punto por cada $10 de compra

In [6]:
from datetime import datetime, date

# 1. Crear instancias de clientes
cliente1 = Cliente("María López", "C001", "555-1234")
cliente2 = Cliente("Carlos Ruiz", "C002", "555-5678")

# 2. Crear bebidas y postres
cafe_americano = Bebida("Café Americano", 2.50, "Café negro tradicional", 
                       "mediano", "caliente", True)
latte = Bebida("Latte", 3.50, "Café con leche cremosa", 
               "grande", "caliente", True)
frappe = Bebida("Frappe", 4.00, "Bebida helada", 
                "grande", "frio", True)

torta_chocolate = Postre("Torta de Chocolate", 4.50, "Torta casera de chocolate", 
                        False, False)
torta_chocolate.ingredientes = ["harina", "huevos", "leche", "chocolate"]

cheesecake = Postre("Cheesecake", 5.00, "Tarta de queso", 
                    False, True)
cheesecake.ingredientes = ["queso crema", "huevos", "azúcar"]

# 3. Configurar inventario
inventario = Inventario()
inventario.ingredientes = {
    "cafe": 100,
    "leche": 50,
    "chocolate": 30,
    "harina": 20,
    "azúcar": 40
}

# 4. Crear empleados
barista = Empleado("Ana García", "E001", "555-9012", "barista", "mañana", 1500.0)
cajero = Empleado("Pedro Martínez", "E002", "555-3456", "cajero", "tarde", 1300.0)

# 5. Crear promociones
promo_puntos = Promocion("PUNTOS20", 0.20, date.today(), date(2025, 12, 31), 100)
promo_especial = Promocion("VERANO10", 0.10, date.today(), date(2025, 8, 31), 50)

# 6. Simular operaciones del sistema
print("=== Simulación del Sistema de Restaurante ===")

# 6.1 Personalizar bebidas
print("\n1. Personalización de bebidas:")
latte.personalizar_bebida(["leche_almendra", "extra_shot"])
precio_final_latte = latte.calcular_precio_final()
print(f"Precio del Latte personalizado: ${precio_final_latte}")

# 6.2 Verificar alérgenos en postres
print("\n2. Verificación de alérgenos:")
alergenos_torta = torta_chocolate.verificar_alergenos()
print(f"Alérgenos en Torta de Chocolate: {alergenos_torta}")

# 6.3 Crear y procesar pedido para cliente1
print("\n3. Procesamiento de pedido:")
pedido1 = Pedido(cliente1)
pedido1.agregar_producto(latte)
pedido1.agregar_producto(torta_chocolate)
total_pedido1 = pedido1.calcular_total()
print(f"Total del pedido: ${total_pedido1}")

# 6.4 Aplicar promoción
cliente1.puntos_fidelidad = 100  # Simulamos que el cliente tiene puntos
if promo_puntos.aplicar_descuento(pedido1):
    print(f"Promoción aplicada. Nuevo total: ${pedido1.total}")

# 6.5 Gestionar estado del pedido
print("\n4. Gestión de estados del pedido:")
barista.gestionar_pedidos(pedido1, EstadoPedido.EN_PREPARACION)
print(f"Estado del pedido: {pedido1.estado.value}")

# 6.6 Actualizar inventario
print("\n5. Gestión de inventario:")
print("Stock inicial de café:", inventario.ingredientes["cafe"])
barista.actualizar_inventario(inventario, "cafe", -2)  # Usar 2 unidades de café
print("Stock después de preparar el pedido:", inventario.ingredientes["cafe"])

# 6.7 Verificar stock bajo
print("\n6. Verificación de stock bajo:")
productos_bajos = inventario.alerta_stock_bajo()
print(f"Productos con stock bajo: {productos_bajos}")

# 6.8 Consultar historial de cliente
print("\n7. Historial de cliente:")
historial = cliente1.consultar_historial()
print(f"Número de pedidos del cliente: {len(historial)}")

# 6.9 Actualizar datos de cliente
print("\n8. Actualización de datos:")
cliente1.actualizar_datos(telefono="555-9999")
print(f"Nuevo teléfono del cliente: {cliente1.telefono}")

# 6.10 Finalizar pedido
barista.gestionar_pedidos(pedido1, EstadoPedido.ENTREGADO)
print(f"\nEstado final del pedido: {pedido1.estado.value}")

# Mostrar puntos de fidelidad ganados
puntos_ganados = promo_puntos.calcular_puntos(pedido1.total)
cliente1.puntos_fidelidad += puntos_ganados
print(f"Puntos de fidelidad actuales: {cliente1.obtener_puntos()}")

=== Simulación del Sistema de Restaurante ===

1. Personalización de bebidas:
Precio del Latte personalizado: $5.0

2. Verificación de alérgenos:
Alérgenos en Torta de Chocolate: ['huevos', 'leche']

3. Procesamiento de pedido:
Total del pedido: $8.0
Promoción aplicada. Nuevo total: $6.4

4. Gestión de estados del pedido:
Estado del pedido: en_preparacion

5. Gestión de inventario:
Stock inicial de café: 100
Stock después de preparar el pedido: 98

6. Verificación de stock bajo:
Productos con stock bajo: []

7. Historial de cliente:
Número de pedidos del cliente: 0

8. Actualización de datos:
Nuevo teléfono del cliente: 555-9999

Estado final del pedido: entregado
Puntos de fidelidad actuales: 100
